In [1]:
import numpy as np

# 흑백 이미지의 픽셀 값
image = np.array([
    [0, 64, 128, 255],
    [0, 128, 192, 255],
    [0, 64, 128, 255]
], dtype=np.uint8)

print("Shape:", image.shape)
print("자료형:", image.dtype)
print("픽셀 값:", image[1, 2])

Shape: (3, 4)
자료형: uint8
픽셀 값: 192


In [3]:
import numpy as np
import torch

# RGB 이미지: 세로 2픽셀, 가로 3픽셀
image = np.array([
    [[255, 0, 0], [0, 255, 0], [0, 0, 255]],
    [[255, 255, 255], [0, 0, 0], [255, 255, 0]]
], dtype=np.uint8)

# NumPy 이미지 구조
print("NumPy Shape:", image.shape)

# NumPy → PyTorch 텐서
tensor = torch.from_numpy(image)

# (H, W, C) → (C, H, W)
tensor = tensor.permute(2, 0, 1)

print("채널 순서 변경:", tensor.shape)

# 배치 차원 추가
tensor = tensor.unsqueeze(0)
# 배치 차원 제거
tensor = tensor.squeeze(0)
print("최종 텐서 Shape:", tensor.shape)

NumPy Shape: (2, 3, 3)
채널 순서 변경: torch.Size([3, 2, 3])
최종 텐서 Shape: torch.Size([3, 2, 3])


In [4]:
from PIL import Image
from torchvision.models import ResNet18_Weights

# 예시 RGB 이미지 생성
image = Image.new(
    "RGB",
    (640, 480),
    color=(120, 170, 200)
)

# 사전학습 모델의 권장 전처리
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()

# 전처리 후 배치 차원 추가
x = preprocess(image).unsqueeze(0)

print("원본 크기:", image.size)
print("입력 Shape:", x.shape)
print("자료형:", x.dtype)

원본 크기: (640, 480)
입력 Shape: torch.Size([1, 3, 224, 224])
자료형: torch.float32


In [5]:
from PIL import Image, ImageDraw
from torchvision import transforms

# 1. 예제 RGB 이미지 생성
image = Image.new(
    "RGB",
    (320, 320),
    color="white"
)

draw = ImageDraw.Draw(image)
draw.rectangle(
    (30, 60, 130, 180),
    fill="red"
)

# 2. 이미지 전처리 구성
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

# 3. 전처리 적용
x = transform(image)

# 4. 결과 확인
print("원본 이미지 크기:", image.size)
print("변환 후 Shape:", x.shape)
print("자료형:", x.dtype)
print("픽셀 최솟값:", x.min().item())
print("픽셀 최댓값:", x.max().item())

원본 이미지 크기: (320, 320)
변환 후 Shape: torch.Size([3, 224, 224])
자료형: torch.float32
픽셀 최솟값: -1.0
픽셀 최댓값: 1.0


In [6]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# 1. 예제 이미지 데이터 구성
records = []

for product_id in range(1, 11):
    label = "불량" if product_id <= 5 else "정상"

    for view in ["front", "side"]:
        records.append({
            "image": f"product_{product_id}_{view}.jpg",
            "product_id": product_id,
            "label": label
        })

df = pd.DataFrame(records)

# 2. Train+Validation / Test 분할
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_val_idx, test_idx = next(
    splitter.split(df, groups=df["product_id"])
)

train_val_df = df.iloc[train_val_idx]
test_df = df.iloc[test_idx]

# 3. Train / Validation 분할
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_val_df,
        groups=train_val_df["product_id"]
    )
)

train_df = train_val_df.iloc[train_idx]
val_df = train_val_df.iloc[val_idx]

# 4. 분할 결과 확인
for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(name, len(data))
    print("제품 ID:", sorted(data["product_id"].unique()))
    print("클래스:", data["label"].value_counts().to_dict())

# 5. 동일 제품의 중복 포함 여부 확인
train_ids = set(train_df["product_id"])
val_ids = set(val_df["product_id"])
test_ids = set(test_df["product_id"])

assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

print("제품 ID 중복 없음")

Train 12
제품 ID: [np.int64(1), np.int64(4), np.int64(5), np.int64(6), np.int64(8), np.int64(10)]
클래스: {'불량': 6, '정상': 6}
Validation 4
제품 ID: [np.int64(3), np.int64(7)]
클래스: {'불량': 2, '정상': 2}
Test 4
제품 ID: [np.int64(2), np.int64(9)]
클래스: {'불량': 2, '정상': 2}
제품 ID 중복 없음


In [1]:
import torch
import torch.nn as nn

# 1. RGB 이미지 1장
x = torch.randn(1, 3, 32, 32)

# 2. CNN 레이어 정의
conv1 = nn.Conv2d(
    in_channels=3,
    out_channels=8,
    kernel_size=3,
    padding=1
)

pool = nn.MaxPool2d(
    kernel_size=2
)

conv2 = nn.Conv2d(
    in_channels=8,
    out_channels=16,
    kernel_size=3,
    padding=1
)

# 3. 특징 추출
feature1 = torch.relu(conv1(x))
feature2 = pool(feature1)
feature3 = torch.relu(conv2(feature2))

# 4. 결과 확인
print("입력:", x.shape)
print("첫 번째 합성곱:", feature1.shape)
print("풀링:", feature2.shape)
print("두 번째 합성곱:", feature3.shape)

입력: torch.Size([1, 3, 32, 32])
첫 번째 합성곱: torch.Size([1, 8, 32, 32])
풀링: torch.Size([1, 8, 16, 16])
두 번째 합성곱: torch.Size([1, 16, 16, 16])


In [2]:
import torch
import torch.nn as nn

# 1. RGB 이미지 1장
x = torch.randn(1, 3, 32, 32)

# 2. CNN 레이어 정의
conv1 = nn.Conv2d(
    in_channels=3,
    out_channels=8,
    kernel_size=3,
    padding=1
)

pool = nn.MaxPool2d(
    kernel_size=2
)

conv2 = nn.Conv2d(
    in_channels=8,
    out_channels=16,
    kernel_size=3,
    padding=1
)

# 3. 특징 추출
feature1 = torch.relu(conv1(x))
feature2 = pool(feature1)
feature3 = torch.relu(conv2(feature2))

# 4. 결과 확인
print("입력:", x.shape)
print("첫 번째 합성곱:", feature1.shape)
print("풀링:", feature2.shape)
print("두 번째 합성곱:", feature3.shape)

입력: torch.Size([1, 3, 32, 32])
첫 번째 합성곱: torch.Size([1, 8, 32, 32])
풀링: torch.Size([1, 8, 16, 16])
두 번째 합성곱: torch.Size([1, 16, 16, 16])


In [3]:
import torch 
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

# 새로운 분류 문제: 정상 / 불량
num_classes = 2

# 1. Scratch: 사전학습 가중치 없이 모델 생성
scratch_model = resnet18(weights=None)

scratch_model.fc = nn.Linear(
    scratch_model.fc.in_features,
    num_classes
)

# 2. Pre-trained: 사전학습 가중치를 불러와 모델 생성
pretrained_model = resnet18(
    weights=ResNet18_Weights.DEFAULT
)

# 새로운 분류 문제에 맞게 최종 분류기 변경
pretrained_model.fc = nn.Linear(
    pretrained_model.fc.in_features,
    num_classes
)

# 3. 분류기 확인
print("Scratch 분류기:", scratch_model.fc)
print("Pre-trained 분류기:", pretrained_model.fc)

# 4. 첫 번째 합성곱 레이어의 가중치 비교
same_weights = torch.equal(
    scratch_model.conv1.weight,
    pretrained_model.conv1.weight
)

print("첫 번째 합성곱 가중치가 동일한가?", same_weights)

Scratch 분류기: Linear(in_features=512, out_features=2, bias=True)
Pre-trained 분류기: Linear(in_features=512, out_features=2, bias=True)
첫 번째 합성곱 가중치가 동일한가? False


In [4]:
print(scratch_model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [5]:
import torch
import torch.nn as nn

torch.manual_seed(42)

# 1. 이미지 분류 모델 정의
model = nn.Sequential(
    nn.Conv2d(3, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(8, 3)
)

# 2. RGB 이미지 4장 생성
X = torch.randn(4, 3, 224, 224)

# 3. 모델 예측
logits = model(X)

# 4. 예측 클래스 확인
predictions = logits.argmax(dim=1)

print("입력 Shape:", X.shape)
print("출력 Shape:", logits.shape)
print("예측 클래스:", predictions)

입력 Shape: torch.Size([4, 3, 224, 224])
출력 Shape: torch.Size([4, 3])
예측 클래스: tensor([1, 1, 1, 1])


In [6]:
import torch

# 1. 클래스 이름
classes = ["신발", "가방", "의류"]

# 2. 이미지 3장에 대한 모델 출력(Logit)
logits = torch.tensor([
    [2.0, 1.0, 0.1],
    [0.2, 2.5, 0.5],
    [1.5, 1.3, 1.0]
])

# 3. Softmax로 예측 확률 계산
probs = torch.softmax(logits, dim=1)

# 4. 예측 클래스 확인
predictions = probs.argmax(dim=1)

# 5. 이미지별 예측 결과 출력
for i in range(len(logits)):
    pred_idx = predictions[i].item()
    pred_label = classes[pred_idx]
    pred_prob = probs[i, pred_idx].item()

    print(f"이미지 {i + 1}")
    print(f"예측 클래스: {pred_label}")
    print(f"예측 확률: {pred_prob:.2%}")
    print()

이미지 1
예측 클래스: 신발
예측 확률: 65.90%

이미지 2
예측 클래스: 가방
예측 확률: 80.93%

이미지 3
예측 클래스: 신발
예측 확률: 41.23%



In [1]:
# 1. 객체 탐지 결과 예시
detections = [
    {
        "class": "상자",
        "bbox": (50, 80, 210, 240),
        "confidence": 0.92
    },
    {
        "class": "상자",
        "bbox": (260, 100, 420, 290),
        "confidence": 0.78
    },
    {
        "class": "팔레트",
        "bbox": (10, 15, 600, 450),
        "confidence": 0.32
    }
]

# 2. Confidence 기준 설정
threshold = 0.5

# 3. 기준을 만족하는 객체 확인
for obj in detections:

    if obj["confidence"] < threshold:
        continue

    x1, y1, x2, y2 = obj["bbox"]

    width = x2 - x1
    height = y2 - y1

    print("클래스:", obj["class"])
    print("Confidence:", obj["confidence"])
    print("Bounding Box:", obj["bbox"])
    print("너비:", width)
    print("높이:", height)
    print()

클래스: 상자
Confidence: 0.92
Bounding Box: (50, 80, 210, 240)
너비: 160
높이: 160

클래스: 상자
Confidence: 0.78
Bounding Box: (260, 100, 420, 290)
너비: 160
높이: 190



In [2]:
# 1. Bounding Box 준비
# (x1, y1, x2, y2)
true_box = (50, 50, 150, 150)
pred_box = (70, 50, 170, 150)

# 2. 겹치는 영역의 좌표 계산
x1 = max(true_box[0], pred_box[0])
y1 = max(true_box[1], pred_box[1])
x2 = min(true_box[2], pred_box[2])
y2 = min(true_box[3], pred_box[3])

# 3. 겹치는 영역의 면적
inter_w = max(0, x2 - x1)
inter_h = max(0, y2 - y1)
intersection = inter_w * inter_h

# 4. 각 Bounding Box의 면적
true_area = (
    (true_box[2] - true_box[0])
    * (true_box[3] - true_box[1])
)

pred_area = (
    (pred_box[2] - pred_box[0])
    * (pred_box[3] - pred_box[1])
)

# 5. IoU 계산
union = true_area + pred_area - intersection
iou = intersection / union

print(f"IoU: {iou:.3f}")

IoU: 0.667


In [3]:
# 1. 객체 탐지 평가 결과
tp = 2
fp = 3
fn = 2

# 2. Precision 계산
precision = tp / (tp + fp)

# 3. Recall 계산
recall = tp / (tp + fn)

# 4. 결과 확인
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")

Precision: 0.40
Recall: 0.50


In [4]:
import numpy as np

# 1. 정답 Mask
true_mask = np.array([
    [0, 0, 0, 0],
    [0, 1, 1, 0],
    [0, 1, 1, 0],
    [0, 0, 0, 0]
])

# 2. 예측 Mask
pred_mask = np.array([
    [0, 0, 0, 0],
    [0, 0, 1, 1],
    [0, 0, 1, 1],
    [0, 0, 0, 0]
])

# 3. 겹치는 영역과 합집합 계산
intersection = np.logical_and(
    true_mask, pred_mask
).sum()

union = np.logical_or(
    true_mask, pred_mask
).sum()

# 4. Mask IoU 계산
iou = intersection / union

# 5. Dice Score 계산
dice = (
    2 * intersection
    / (true_mask.sum() + pred_mask.sum())
)

# 6. 결과 확인
print(f"Mask IoU: {iou:.3f}")
print(f"Dice Score: {dice:.3f}")

Mask IoU: 0.333
Dice Score: 0.500


In [5]:
import pandas as pd

# 1. 오류 사례 기록
records = [
    {"image": "A.jpg", "task": "분류", "error": "오분류"},
    {"image": "B.jpg", "task": "분류", "error": "오분류"},
    {"image": "C.jpg", "task": "객체 탐지", "error": "미검출"},
    {"image": "D.jpg", "task": "객체 탐지", "error": "미검출"},
    {"image": "E.jpg", "task": "객체 탐지", "error": "잘못된 검출"},
    {"image": "F.jpg", "task": "세그멘테이션", "error": "영역 누락"},
    {"image": "G.jpg", "task": "세그멘테이션", "error": "영역 과다 예측"}
]

df = pd.DataFrame(records)

# 2. 태스크별 오류 유형과 건수 확인
error_counts = (
    df.groupby(["task", "error"])
    .size()
    .reset_index(name="count")
)

print(error_counts)

     task     error  count
0   객체 탐지       미검출      2
1   객체 탐지    잘못된 검출      1
2      분류       오분류      2
3  세그멘테이션  영역 과다 예측      1
4  세그멘테이션     영역 누락      1


In [6]:
import pandas as pd

# 1. 개선 전후 평가 결과
results = pd.DataFrame({
    "Metric": ["Precision", "Recall", "mAP@0.5"],
    "Before": [0.85, 0.60, 0.68],
    "After": [0.78, 0.78, 0.74]
})

# 2. 개선 전후 변화량 계산
results["Change"] = (
    results["After"] - results["Before"]
)

# 3. 결과 확인
print(results.round(2))

      Metric  Before  After  Change
0  Precision    0.85   0.78   -0.07
1     Recall    0.60   0.78    0.18
2    mAP@0.5    0.68   0.74    0.06


In [7]:
import numpy as np

# 1. 이미지 이름
images = ["운동화 A", "운동화 B", "가방 C"]

# 2. 이미지별 임베딩 벡터 예시
embeddings = np.array([
    [0.80, 0.15, 0.60, 0.25],
    [0.78, 0.18, 0.61, 0.22],
    [0.12, 0.85, 0.18, 0.76]
])

# 3. 임베딩 형태 확인
print("이미지 수:", embeddings.shape[0])
print("임베딩 차원:", embeddings.shape[1])
print("전체 Shape:", embeddings.shape)

# 4. 이미지별 임베딩 출력
for image, vector in zip(images, embeddings):
    print(image, vector)

이미지 수: 3
임베딩 차원: 4
전체 Shape: (3, 4)
운동화 A [0.8  0.15 0.6  0.25]
운동화 B [0.78 0.18 0.61 0.22]
가방 C [0.12 0.85 0.18 0.76]
